#### マッピング

In [1]:
from pathlib import Path
import zipfile
import geopandas as gpd
import folium
import branca.colormap as cm
import yaml

# =========================================================
# 0. config + fallback
# =========================================================
CONFIG_PATH = Path("config.yaml")

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    DATA_DIR = Path(config["data_root"]).expanduser()

    if "project_root" in config:
        PROJECTS_DIR = Path(config["project_root"]).expanduser()
    else:
        PROJECTS_DIR = DATA_DIR.parent / "Projects"

else:
    DATA_DIR = Path("~/Desktop/Research/Data").expanduser()
    PROJECTS_DIR = Path("~/Desktop/Research/Projects").expanduser()

RESEARCH_DIR = DATA_DIR.parent

print("DATA_DIR:", DATA_DIR)
print("PROJECTS_DIR:", PROJECTS_DIR)

# =========================================================
# LandUse
# =========================================================
LANDUSE_RAW_DIR = DATA_DIR / "Raw" / "LandUse" / "2021" / "national_landuse"
LANDUSE_INTERMEDIATE_DIR = DATA_DIR / "Intermediate" / "LandUse" / "2021"
LANDUSE_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "landuse_2021"
LANDUSE_PROCESSED_DIR = DATA_DIR / "Processed" / "LandUse" / "2021"

# =========================================================
# ZIP polygon（展開済みshape）
# =========================================================
ZIP_RAW_DIR = DATA_DIR / "Raw" / "Geospatial" / "ZIP_polygon" / "2022" / "2022_GEOSPACE郵便番号ポリゴン"
ZIP_SHAPE_DIR = ZIP_RAW_DIR / "shape"
ZIP_SHP_PATH = ZIP_SHAPE_DIR / "GSP99.shp"
ZIP_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "zipcode_polygon_2022"

# =========================================================
# Mesh
# =========================================================
MESH_RAW_DIR = DATA_DIR / "Raw" / "Geospatial" / "Mesh" / "500m_2024"
MESH_RAW_INNER_DIR = MESH_RAW_DIR / "500m_mesh_2024_GEOJSON"

MESH_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "mesh_500m_2024"
MESH_GEOJSON_DIR = MESH_EXTRACT_DIR / "geojson"

# =========================================================
# Project output
# =========================================================
LANDUSE_BUILD_DIR = PROJECTS_DIR / "LandUse_build"
LANDUSE_MAP_DIR = LANDUSE_BUILD_DIR / "output" / "maps"

# フォルダ作成
for p in [
    LANDUSE_EXTRACT_DIR,
    LANDUSE_PROCESSED_DIR,
    ZIP_EXTRACT_DIR,
    MESH_EXTRACT_DIR,
    MESH_GEOJSON_DIR,
    LANDUSE_MAP_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

# =========================================================
# 出力パス
# =========================================================
PROCESSED_DIR = LANDUSE_PROCESSED_DIR
gpkg_path = PROCESSED_DIR / "zipcode_landuse_area.gpkg"

# =========================================================
# 確認
# =========================================================
print("gpkg_path:", gpkg_path)
print("  exists:", gpkg_path.exists())

print("ZIP_SHP_PATH:", ZIP_SHP_PATH)
print("  exists:", ZIP_SHP_PATH.exists())

gpkg_path: /Users/Tomo/Desktop/Research/Data/Processed/LandUse/2021/zipcode_landuse_area.gpkg
  exists: True
ZIP_SHP_PATH: /Users/Tomo/Desktop/Research/Data/Raw/Geospatial/ZIP_polygon/2022/2022_GEOSPACE郵便番号ポリゴン/shape/GSP99.shp
  exists: True


In [ ]:
zipcode_gdf = gpd.read_file(gpkg_path)

map_gdf = zipcode_gdf[[
    "zipcode",
    "population_2020",
    "residential_area_m2",
    "pop_density_residential",
    "geometry"
]].copy()

map_gdf = map_gdf.dropna(subset=["pop_density_residential"]).copy()
map_gdf["zipcode"] = map_gdf["zipcode"].astype(str).str.zfill(7)
map_gdf = map_gdf.to_crs(epsg=4326)

# 地図中心
centroid = map_gdf.geometry.centroid
center_lat = centroid.y.mean()
center_lon = centroid.x.mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=5,
    tiles="CartoDB positron"
)

min_val = map_gdf["pop_density_residential"].min()
max_val = map_gdf["pop_density_residential"].max()

original_colormap = cm.linear.Spectral_11
reversed_colors = list(original_colormap.colors)[::-1]
colormap = cm.LinearColormap(reversed_colors, vmin=min_val, vmax=max_val)
colormap.caption = "Population Density on Residential Area"

folium.GeoJson(
    map_gdf,
    name="Residential Population Density",
    style_function=lambda feature: {
        "fillColor": colormap(feature["properties"]["pop_density_residential"])
        if feature["properties"]["pop_density_residential"] is not None else "transparent",
        "color": None,
        "weight": 0,
        "fillOpacity": 0.7
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["zipcode", "population_2020", "residential_area_m2", "pop_density_residential"],
        aliases=["Zipcode", "Population 2020", "Residential area (m2)", "Density (per km2)"],
        localize=True
    )
).add_to(m)

colormap.add_to(m)

out_html = LANDUSE_MAP_DIR / "pop_density_residential_folium.html"
m.save(str(out_html))

print("Saved:", out_html)
m

/var/folders/lx/_3z8wwyx3ds15_0k28hby_h80000gn/T/ipykernel_34130/993986728.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = map_gdf.geometry.centroid


Saved: /Users/Tomo/Desktop/Research/Projects/LandUse_build/output/maps/pop_density_residential_folium.html
